# Deep Learning Literature and Sequence EDA

**Owner:** Prateek Shakya

## Purpose
Review sequence-learning literature and quantify the sequence properties that determine windowing choices.

## Method
The notebook uses only the repository data and records unreported paper-specific values as `n/a`. All EDA is descriptive; validation and test labels are not used to fit a model.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from load_data import load_train, load_test, SENSORS
from evaluation import split_train_val, validation_points

FIG = ROOT / "results/phase2/figures"
FIG.mkdir(parents=True, exist_ok=True)
train, test = load_train(), load_test()
train_fit, val = split_train_val(train)
points = validation_points(val)


### Engine lengths
This cell compares run lengths and counts short test engines. Left padding makes every observed row usable even when the history is shorter than the requested window.

In [ ]:
lengths = pd.DataFrame({"train": train.groupby("engine_id").size(), "test": test.groupby("engine_id").size()})
print(lengths.describe().loc[["min","mean","max"]])
print({w: int((lengths["test"] < w).sum()) for w in [20,30,40,50]})
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(lengths["train"], bins=15, alpha=.65, label="train")
ax.hist(lengths["test"], bins=15, alpha=.65, label="test")
ax.set(xlabel="Observed cycles", ylabel="Number of engines", title="FD001 engine length distribution")
ax.legend(); fig.tight_layout(); fig.savefig(FIG/"eda_engine_lengths.png", dpi=160); plt.show()


### Sensor trajectories
The x-axis is cycles before failure, so trajectories from engines with different run lengths are aligned at failure. These examples show why trend information is useful but not uniform across sensors.

In [ ]:
chosen = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11"]
fig, axes = plt.subplots(len(chosen), 1, figsize=(8, 11), sharex=True)
for ax, sensor in zip(axes, chosen):
    for eid in sorted(train.engine_id.unique())[:12]:
        g = train[train.engine_id == eid]
        failure = g.cycle.max()
        ax.plot(failure-g.cycle, g[sensor], alpha=.55)
    ax.set_ylabel(sensor)
axes[-1].set_xlabel("Cycles before failure")
fig.suptitle("Representative FD001 sensor trajectories (12 engines)")
fig.tight_layout(); fig.savefig(FIG/"eda_sensor_trajectories.png", dpi=160); plt.show()


### Operating conditions, targets, and normalization
FD001 is treated as one operating regime because its settings have very small variation compared with the degradation trajectories. The target distribution also shows why uncapped early-life points are difficult: a model trained mostly near failure has little evidence for very large RUL.

In [ ]:
print("Setting ranges:")
print(train[["setting_1","setting_2","setting_3"]].agg(["min","max","std"]).T)
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].hist(train.rul, bins=30, alpha=.75); axes[0].set(title="All training RUL", xlabel="RUL (cycles)", ylabel="Rows")
axes[1].hist(points.rul, bins=20, alpha=.75); axes[1].set(title="100 validation points", xlabel="RUL (cycles)", ylabel="Points")
fig.tight_layout(); fig.savefig(FIG/"eda_rul_distributions.png", dpi=160); plt.show()
scaler_train = train_fit[SENSORS].agg(["mean","std"]).T
print(scaler_train)
fig, ax = plt.subplots(figsize=(8,4)); scaler_train["std"].plot.bar(ax=ax); ax.set(xlabel="Sensor", ylabel="Training standard deviation", title="Training-only sensor scales"); fig.tight_layout(); fig.savefig(FIG/"eda_sensor_scales.png", dpi=160); plt.show()


## Results
The generated figures and printed summaries provide the engine-length, trajectory, operating-condition, target, and normalization evidence required by the sequence pipeline. The literature table is copied from `results/phase2/literature_table.csv` and uses `n/a` where a paper-specific number was not verified.

In [ ]:
literature = pd.read_csv(ROOT / "results/phase2/literature_table.csv")
literature


## Conclusion
Sequence windows retain order and multiple cycles, unlike flat rolling summaries whose history length and statistics are fixed in advance. The small number of engines makes regularization, engine-level splitting, and sensitivity to window length central concerns. These gaps motivate later recurrent, convolutional, attention, and hybrid comparisons.

## Handoff
This notebook produces `results/phase2/literature_table.csv` and the `eda_*.png` figures. The next role should use `src/sequences.py` with the training-only scaler and the fixed engine split.